# 🎯 DealHunter — Milestone 7

---

### What is DealHunter?
DealHunter is a **product price prediction system** that benchmarks multiple LLMs on estimating retail prices from product titles.

**Models compared:**
- ✅ Gemini 2.5 Flash (Google — FREE)
- ✅ Llama 3.3 70B via Groq (META/Groq — FREE)
- ⬜ GPT-4o-mini (OpenAI — optional, paid)

**Metrics:** MAE (Mean Absolute Error $) and MAPE (Mean Absolute % Error)

---

### FREE API Keys
| Provider | URL | Daily Limit |
|----------|-----|-------------|
| Google Gemini | https://aistudio.google.com/app/apikey | 1M tokens/day |
| Groq (Llama) | https://console.groq.com | 14,400 req/day |

---

### Run Order
Run **Cell 1 → Cell 2 → Cell 3 → Cell 4 → ... → Cell 13** in order.
**Do NOT skip any cell.**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 1 — Environment Check
# ============================================================
# Verifies Python version and GPU availability.
# Expected output: Python version + Colab confirmation
# ============================================================

import sys, subprocess

print(f'Python: {sys.version}')

try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    print(f'GPU: {r.stdout.strip()}' if r.returncode==0 else 'No GPU (CPU only — fine for this notebook)')
except Exception:
    print('GPU check skipped')

try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab ✅')
except ImportError:
    IN_COLAB = False
    print('Running locally')

print('\nCell 1 complete ✅')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
GPU check skipped
Running in Google Colab ✅

Cell 1 complete ✅


In [4]:
# ============================================================
# CELL 2 — Install Dependencies
# ============================================================
# Installs all required packages with pinned compatible versions.
# IMPORTANT: httpx is pinned to 0.27.2 to fix the Groq
#   'proxies' error that occurs with httpx >= 0.28.
# Expected output: Successfully installed ... messages
# Runtime: 2-4 minutes on first run
# ============================================================

# Pin httpx FIRST before anything else imports it
import subprocess, sys

packages = [
    'httpx==0.27.2',                 # MUST be first — fixes Groq proxies error
    'google-generativeai>=0.7.2',
    'groq>=0.9.0',
    'openai>=1.30.1',
    'datasets>=2.19.1',
    'pandas',
    'numpy',
    'scikit-learn',
    'tqdm',
    'plotly',
    'streamlit>=1.35.0',
    'pyngrok',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\n✅ All packages installed!')
print('⚠️  If you see any ❌ above, re-run this cell once more.')

✅ httpx==0.27.2
✅ google-generativeai>=0.7.2
✅ groq>=0.9.0
✅ openai>=1.30.1
✅ datasets>=2.19.1
✅ pandas
✅ numpy
✅ scikit-learn
✅ tqdm
✅ plotly
❌ streamlit>=1.35.0
✅ pyngrok

✅ All packages installed!
⚠️  If you see any ❌ above, re-run this cell once more.


In [ ]:
# ============================================================
# CELL 3 — API Key Configuration
# ============================================================
# Set your FREE API keys here.
#
# GET KEYS (both are FREE):
#   Gemini : https://aistudio.google.com/app/apikey
#   Groq   : https://console.groq.com
#
# SAFER OPTION — Use Colab Secrets (🔑 icon in left sidebar):
#   Add secrets named GOOGLE_API_KEY and GROQ_API_KEY
#   Then uncomment the userdata lines below.
#
# Expected output: ✅ SET for each key you provided
# ============================================================

import os

# ── Option A: Enter directly ─────────────────────────────────
GOOGLE_API_KEY = ''
GROQ_API_KEY   = ''
OPENAI_API_KEY = ''   # optional — leave blank to skip

# ── Option B: Colab Secrets (recommended) ────────────────────
# from google.colab import userdata
# GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
# GROQ_API_KEY   = userdata.get('GROQ_API_KEY')

# Set env vars
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
os.environ['GROQ_API_KEY']   = GROQ_API_KEY
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Validate
def _is_set(key, placeholder):
    return key and key != placeholder and len(key) > 10

print('API Key Status:')
print(f'  Google Gemini : {"✅ SET" if _is_set(GOOGLE_API_KEY, "YOUR_GOOGLE_API_KEY_HERE") else "❌ NOT SET — get free key at aistudio.google.com"}')
print(f'  Groq (Llama)  : {"✅ SET" if _is_set(GROQ_API_KEY,   "YOUR_GROQ_API_KEY_HERE")   else "❌ NOT SET — get free key at console.groq.com"}')
print(f'  OpenAI        : {"✅ SET" if OPENAI_API_KEY else "⏭  Skipped (optional)"}')
print('\nCell 3 complete ✅')

API Key Status:
  Google Gemini : ✅ SET
  Groq (Llama)  : ✅ SET
  OpenAI        : ⏭  Skipped (optional)

Cell 3 complete ✅


In [6]:
# ============================================================
# CELL 4 — Core Imports & Project Setup
# ============================================================
# Imports all libraries and creates the project folder
# structure under /content/DealHunter/
# Expected output: directory tree + 'Imports complete'
# ============================================================

import re, json, time, random
from pathlib import Path
from typing  import Optional

import numpy  as np
import pandas as pd
from tqdm.notebook import tqdm
import plotly.express    as px
import plotly.graph_objects as go
from IPython.display import display

# Project directories
BASE_DIR   = Path('/content/DealHunter')
OUTPUT_DIR = BASE_DIR / 'outputs'
for d in [BASE_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project structure:')
print(f'  {BASE_DIR}/')
print(f'    outputs/')
print('\nImports complete ✅')

Project structure:
  /content/DealHunter/
    outputs/

Imports complete ✅


In [7]:
# ============================================================
# CELL 5 — Data Loading & Feature Engineering
# ============================================================
# Loads Amazon Electronics product data from HuggingFace.
# Falls back to realistic synthetic data if network is slow.
# Adds engineered features: clean title, inferred category,
# word count, log price, price bucket.
#
# Expected output:
#   - N products loaded
#   - Price statistics table
#   - df.head() with 10 sample rows
# ============================================================

MAX_PRODUCTS = 300

CATEGORY_MAP = {
    'electronics': ['phone','laptop','tablet','camera','headphone','speaker',
                    'monitor','keyboard','mouse','tv','charger','cable','ipad'],
    'clothing'   : ['shirt','dress','shoes','pants','jacket','jeans','boots','sneaker'],
    'home'       : ['furniture','lamp','chair','table','bed','sofa','rug','cookware','vacuum'],
    'beauty'     : ['skincare','makeup','perfume','lipstick','serum','cream','foundation'],
    'sports'     : ['gym','yoga','running','bicycle','fitness','tennis','basketball'],
    'books'      : ['novel','textbook','biography','cookbook','guide','hardcover'],
    'toys'       : ['toy','game','puzzle','lego','doll','action figure'],
}

def infer_category(title: str) -> str:
    t = title.lower()
    for cat, kws in CATEGORY_MAP.items():
        if any(k in t for k in kws):
            return cat
    return 'other'

def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', str(text).strip())
    text = re.sub(r'[^\w\s\-\.,()°%$/]', '', text)
    return text[:200]

def generate_synthetic(n: int = 300) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    TEMPLATES = [
        ('Apple iPhone {v} - {s}GB - {c}',                    'electronics', 799, 200),
        ('Samsung Galaxy S{v} Ultra Smartphone',               'electronics',1099, 300),
        ('Sony WH-1000XM{v} Noise Cancelling Headphones',      'electronics', 349,  80),
        ('Logitech MX Master {v} Wireless Mouse',              'electronics',  79,  20),
        ('Dell XPS {v} 15-inch Laptop Intel Core i7',          'electronics',1399, 400),
        ('Canon EOS R{v} Mirrorless Camera Body',              'electronics',1299, 500),
        ('iPad Pro {v}-inch M2 Chip Wi-Fi',                    'electronics',1099, 250),
        ('Nintendo Switch OLED {c} Edition',                   'electronics', 349,  50),
        ('Anker PowerCore {v}000mAh Portable Charger',         'electronics',  39,  15),
        ('Nike Air Max {v} Running Shoes Men',                 'clothing',    129,  30),
        ("Levi's 501 Original Fit Jeans {c}",                  'clothing',     59,  15),
        ('KitchenAid Artisan Stand Mixer {c}',                 'home',        449, 100),
        ('Dyson V{v} Cordless Vacuum Cleaner',                 'home',        499, 150),
        ('Instant Pot Duo 7-in-1 Electric Pressure Cooker',    'home',         99,  25),
        ('IKEA KALLAX Shelf Unit {c}',                         'home',         89,  20),
        ('Atomic Habits by James Clear Hardcover',             'books',        17,   4),
        ('The Psychology of Money Morgan Housel',              'books',        15,   4),
        ('LEGO Technic {v} Building Set',                      'toys',         89,  40),
        ('Gaiam Premium Yoga Mat {c} 6mm Thick',               'sports',       35,  12),
        ('CeraVe Moisturizing Cream 19oz Daily Face Body',     'beauty',       18,   5),
    ]
    rows = []
    for _ in range(n):
        tmpl, cat, mu, sigma = TEMPLATES[rng.integers(0, len(TEMPLATES))]
        price = float(np.clip(rng.normal(mu, sigma), mu*0.3, mu*2.5))
        title = tmpl.format(
            v=rng.integers(5,16),
            s=rng.choice([64,128,256,512]),
            c=rng.choice(['Black','White','Silver','Blue','Red','Green']),
        )
        rows.append({'title': title, 'price': round(price,2), 'category': cat})
    return pd.DataFrame(rows)

def load_data(max_rows: int = 300) -> pd.DataFrame:
    try:
        from datasets import load_dataset as hf_load
        print('Loading Amazon Electronics from HuggingFace (streaming)...')
        ds = hf_load('McAuley-Lab/Amazon-Reviews-2023', 'raw_meta_Electronics',
                     split='full', streaming=True, trust_remote_code=True)
        rows = []
        for item in ds:
            if len(rows) >= max_rows: break
            price = item.get('price'); title = item.get('title','')
            if not title or not price: continue
            try:
                pv = float(str(price).replace('$','').replace(',','').strip())
            except (ValueError, AttributeError):
                continue
            if 0 < pv <= 5000:
                rows.append({'title': title, 'price': pv, 'category': 'Electronics'})
        if len(rows) < 50: raise ValueError('Too few rows')
        df = pd.DataFrame(rows)
        print(f'✅ Loaded {len(df)} Amazon products')
        return df
    except Exception as e:
        print(f'HuggingFace failed ({e}). Using synthetic data.')
        df = generate_synthetic(max_rows)
        print(f'✅ Generated {len(df)} synthetic products')
        return df

# ── Load ─────────────────────────────────────────────────────
df_raw = load_data(MAX_PRODUCTS)

# ── Feature engineering ──────────────────────────────────────
df = df_raw.copy()
df['title_clean']       = df['title'].apply(clean_text)
df['category_inferred'] = df['title_clean'].apply(infer_category)
df['word_count']        = df['title_clean'].str.split().str.len()
df['log_price']         = np.log1p(df['price'])
df['price_bucket']      = pd.cut(
    df['price'], bins=[0,25,75,200,500,10000],
    labels=['budget','low','mid','high','premium']
)

print(f'\nDataset shape: {df.shape}')
print(f'\nPrice statistics:')
print(df['price'].describe().round(2).to_string())
print(f'\nCategory breakdown:')
print(df['category_inferred'].value_counts().to_string())
print()
display(df[['title_clean','price','category_inferred','price_bucket']].head(10))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'McAuley-Lab/Amazon-Reviews-2023' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading Amazon Electronics from HuggingFace (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Amazon-Reviews-2023.py: 0.00B [00:00, ?B/s]

HuggingFace failed (Dataset scripts are no longer supported, but found Amazon-Reviews-2023.py). Using synthetic data.
✅ Generated 300 synthetic products

Dataset shape: (300, 8)

Price statistics:
count     300.00
mean      340.60
std       460.20
min        10.59
25%        43.84
50%       102.18
75%       455.75
max      2267.62

Category breakdown:
category_inferred
electronics    114
other           64
clothing        40
sports          21
beauty          18
toys            16
books           14
home            13



,title_clean,price,category_inferred,price_bucket
0,Samsung Galaxy S13 Ultra Smartphone,787.00,electronics,premium
1,Samsung Galaxy S12 Ultra Smartphone,513.69,electronics,premium
2,IKEA KALLAX Shelf Unit Black,82.68,other,mid
3,The Psychology of Money Morgan Housel,18.52,other,budget
4,Atomic Habits by James Clear Hardcover,21.51,books,budget
5,Nike Air Max 7 Running Shoes Men,140.06,clothing,mid
6,LEGO Technic 14 Building Set,87.00,toys,mid
7,IKEA KALLAX Shelf Unit Green,113.45,other,mid
8,Instant Pot Duo 7-in-1 Electric Pressure Cooker,90.20,other,mid
9,Nike Air Max 5 Running Shoes Men,141.38,clothing,mid


In [8]:
# ============================================================
# CELL 6 — Exploratory Data Analysis (EDA)
# ============================================================
# Three interactive Plotly charts:
#   1. Price distribution histogram
#   2. Price by category (box plot)
#   3. Word count vs price scatter
# Expected output: 3 charts rendered inline
# ============================================================

# Chart 1 — Price distribution
fig1 = px.histogram(
    df, x='price', nbins=40,
    title='Product Price Distribution',
    labels={'price':'Price (USD)'},
    color_discrete_sequence=['#636EFA'],
)
fig1.update_layout(bargap=0.1)
fig1.show()

# Chart 2 — Price by category
fig2 = px.box(
    df, x='category_inferred', y='price',
    title='Price by Category',
    labels={'category_inferred':'Category','price':'Price (USD)'},
    color='category_inferred',
)
fig2.update_layout(showlegend=False, xaxis_tickangle=-30)
fig2.show()

# Chart 3 — Word count vs price
fig3 = px.scatter(
    df, x='word_count', y='price',
    color='category_inferred',
    title='Title Word Count vs Price',
    labels={'word_count':'Words in Title','price':'Price (USD)'},
    opacity=0.7,
)
fig3.show()

print('EDA complete ✅')

EDA complete ✅


In [9]:
# ============================================================
# CELL 7 — Train / Test Split
# ============================================================
# Splits data 80/20. Evaluation runs on 50 products max
# to stay within free API rate limits.
# Expected output: split sizes + price range of test set
# ============================================================

EVAL_SAMPLES = 50
SEED         = 42

df_shuffled  = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split_idx    = int(len(df_shuffled) * 0.8)
train_df     = df_shuffled.iloc[:split_idx].copy()
test_df      = df_shuffled.iloc[split_idx:].copy().reset_index(drop=True)
test_eval_df = test_df.head(EVAL_SAMPLES).reset_index(drop=True)

print(f'Train : {len(train_df)} products')
print(f'Test  : {len(test_df)} products')
print(f'Eval  : {len(test_eval_df)} products (API budget cap)')
print(f'\nTest price range : ${test_eval_df["price"].min():.2f} — ${test_eval_df["price"].max():.2f}')
print(f'Test mean price  : ${test_eval_df["price"].mean():.2f}')

Train : 240 products
Test  : 60 products
Eval  : 50 products (API budget cap)

Test price range : $13.17 — $2267.62
Test mean price  : $369.38


In [24]:
# ── Debug Cell — run this to diagnose ──
import os

gkey = os.environ.get('GROQ_API_KEY', '')
print(f'GROQ_API_KEY length : {len(gkey)}')
print(f'GROQ_API_KEY value  : {repr(gkey[:20])}...')  # shows first 20 chars only
print(f'Check passes (>10)  : {len(gkey) > 10}')



GROQ_API_KEY length : 56
GROQ_API_KEY value  : 'gsk_h7kmDO3htZvC3tap'...
Check passes (>10)  : True


In [10]:
# ── Force-rebuild GroqClient from scratch ──
import os, re, time
from typing import Optional

class GroqClient:
    """Standalone GroqClient — no inheritance, no ABC."""

    def __init__(self):
        self.model_name   = 'llama-3.3-70b-versatile'
        self.min_interval = 60.0 / 28   # 28 RPM
        self._last_call   = 0.0
        self._n_calls     = 0
        self._total_time  = 0.0
        self._client      = self._build_client()
        print(f'  GroqClient built successfully')

    def _build_client(self):
        from groq import Groq
        try:
            return Groq(api_key=os.environ['GROQ_API_KEY'])
        except TypeError:
            import httpx
            return Groq(
                api_key=os.environ['GROQ_API_KEY'],
                http_client=httpx.Client(),
            )

    def _call_api(self, prompt: str) -> str:
        resp = self._client.chat.completions.create(
            model       = self.model_name,
            messages    = [{'role': 'user', 'content': prompt}],
            temperature = 0.0,
            max_tokens  = 50,
        )
        return resp.choices[0].message.content.strip()

    def predict(self, prompt: str, retries: int = 2) -> Optional[str]:
        wait = self.min_interval - (time.time() - self._last_call)
        if wait > 0:
            time.sleep(wait)
        self._last_call = time.time()

        for attempt in range(retries + 1):
            try:
                t0   = time.perf_counter()
                resp = self._call_api(prompt)
                self._total_time += time.perf_counter() - t0
                self._n_calls    += 1
                return resp
            except Exception as e:
                if attempt < retries:
                    w = 2 ** attempt
                    print(f'  Retry {attempt+1} ({self.model_name}): {e}. Waiting {w}s...')
                    time.sleep(w)
                else:
                    print(f'  FAIL ({self.model_name}): {e}')
                    return None

    @property
    def avg_latency_ms(self):
        return (self._total_time / self._n_calls * 1000) if self._n_calls > 0 else 0.0


# ── Test it immediately ──────────────────────────────────────
try:
    groq_client = GroqClient()
    test_resp = groq_client._call_api(
        'Return only the number 299. No text, no punctuation.'
    )
    print(f'✅ Groq working! Test response: {repr(test_resp)}')

    # Add to CLIENTS list (replace old GroqClient if present)
    CLIENTS = [c for c in CLIENTS if 'llama' not in c.model_name.lower()]
    CLIENTS.append(groq_client)
    print(f'\nActive clients: {[c.model_name for c in CLIENTS]}')

except Exception as e:
    print(f'❌ Groq failed: {e}')


  GroqClient built successfully
✅ Groq working! Test response: '299'
❌ Groq failed: name 'CLIENTS' is not defined


In [16]:
# ── Build CLIENTS from scratch with both working clients ──

CLIENTS = []

# Add Groq (already built and tested)
CLIENTS.append(groq_client)

# Add OpenRouter Gemma (already built and tested)
CLIENTS.append(or_client)

print(f'✅ Active clients ({len(CLIENTS)}):')
for c in CLIENTS:
    print(f'   {c.model_name}')

print('\nRun Cell 10 now ▶')


✅ Active clients (2):
   llama-3.3-70b-versatile
   google/gemma-3-27b-it:free

Run Cell 10 now ▶


In [17]:
# ============================================================
# CELL 9 — Metrics Functions
# ============================================================
# Defines MAE, MAPE, RMSE and accuracy-within-tolerance.
# compute_metrics() has a NaN guard — returns None values
# (not nan) when zero predictions succeeded.
# Expected output: demo metrics on toy data
# ============================================================

def mae(y_true, y_pred):
    """Mean Absolute Error — average dollar error."""
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — scale-invariant, main metric."""
    a, p = np.array(y_true, float), np.array(y_pred, float)
    mask = a != 0
    return float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100)

def rmse(y_true, y_pred):
    """Root Mean Squared Error — penalises large errors."""
    return float(np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2)))

def accuracy_within(y_true, y_pred, tol=20):
    """% of predictions within ±tol% of the true price."""
    a, p = np.array(y_true, float), np.array(y_pred, float)
    return float(np.mean(np.abs(a-p)/np.maximum(a,1e-9)*100 <= tol)*100)

def compute_metrics(y_true, y_pred, model_name, n_failed=0):
    """Compute all metrics with NaN guard for empty results."""
    n_total = len(y_true) + n_failed

    # NaN guard — if ALL predictions failed, return None values
    if len(y_true) == 0:
        print(f'  ⚠️  {model_name}: 0 successful predictions out of {n_total}')
        return {
            'model': model_name, 'n_evaluated': 0, 'n_failed': n_failed,
            'parse_success_rate': 0.0,
            'mae': None, 'mape': None, 'rmse': None,
            'accuracy_within_10pct': None,
            'accuracy_within_20pct': None,
            'accuracy_within_50pct': None,
        }

    return {
        'model':                   model_name,
        'n_evaluated':             len(y_true),
        'n_failed':                n_failed,
        'parse_success_rate':      round(len(y_true)/n_total*100, 2),
        'mae':                     round(mae(y_true, y_pred), 2),
        'mape':                    round(mape(y_true, y_pred), 2),
        'rmse':                    round(rmse(y_true, y_pred), 2),
        'accuracy_within_10pct':   round(accuracy_within(y_true, y_pred, 10), 2),
        'accuracy_within_20pct':   round(accuracy_within(y_true, y_pred, 20), 2),
        'accuracy_within_50pct':   round(accuracy_within(y_true, y_pred, 50), 2),
    }

# ── Quick smoke test ─────────────────────────────────────────
print('Smoke test (near-perfect model):')
t, p = [100,200,50,300,75], [92,210,55,280,80]
m    = compute_metrics(t, p, 'Smoke Test')
print(f'  MAE={m["mae"]}  MAPE={m["mape"]}%  ±20% Acc={m["accuracy_within_20pct"]}%')
print('\nMetric functions ready ✅')

Smoke test (near-perfect model):
  MAE=9.6  MAPE=7.27%  ±20% Acc=100.0%

Metric functions ready ✅


In [19]:
# ── Define all missing functions ──
import re
from typing import Optional

def build_prompt(title: str, category: Optional[str] = None) -> str:
    cat_hint = f' in the {category} category' if category else ''
    return (
        f'You are a product pricing expert. Based on the product title below'
        f'{cat_hint}, estimate its retail price in USD.\n\n'
        f'Product: {title}\n\n'
        f'Rules:\n'
        f'- Return ONLY a number (e.g., 29.99)\n'
        f'- No dollar sign, no text, no explanation\n'
        f'- Base your estimate on typical US retail prices\n\n'
        f'Price:'
    )

def parse_price(response: str) -> Optional[float]:
    if not response:
        return None
    text  = str(response).strip().replace('$','').replace(',','').replace('USD','')
    match = re.search(r'\b(\d{1,5}(?:\.\d{1,2})?)\b', text)
    if match:
        price = float(match.group(1))
        if 0.01 <= price <= 100_000:
            return price
    return None

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def mape(y_true, y_pred):
    a, p = np.array(y_true, float), np.array(y_pred, float)
    mask = a != 0
    return float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2)))

def accuracy_within(y_true, y_pred, tol=20):
    a, p = np.array(y_true, float), np.array(y_pred, float)
    return float(np.mean(np.abs(a-p)/np.maximum(a,1e-9)*100 <= tol)*100)

def compute_metrics(y_true, y_pred, model_name, n_failed=0):
    n_total = len(y_true) + n_failed
    if len(y_true) == 0:
        print(f'  ⚠️  {model_name}: 0 successful predictions out of {n_total}')
        return {
            'model': model_name, 'n_evaluated': 0, 'n_failed': n_failed,
            'parse_success_rate': 0.0,
            'mae': None, 'mape': None, 'rmse': None,
            'accuracy_within_10pct': None,
            'accuracy_within_20pct': None,
            'accuracy_within_50pct': None,
        }
    return {
        'model':                   model_name,
        'n_evaluated':             len(y_true),
        'n_failed':                n_failed,
        'parse_success_rate':      round(len(y_true)/n_total*100, 2),
        'mae':                     round(mae(y_true, y_pred), 2),
        'mape':                    round(mape(y_true, y_pred), 2),
        'rmse':                    round(rmse(y_true, y_pred), 2),
        'accuracy_within_10pct':   round(accuracy_within(y_true, y_pred, 10), 2),
        'accuracy_within_20pct':   round(accuracy_within(y_true, y_pred, 20), 2),
        'accuracy_within_50pct':   round(accuracy_within(y_true, y_pred, 50), 2),
    }

# Confirm everything needed is available
print('✅ build_prompt   defined')
print('✅ parse_price    defined')
print('✅ compute_metrics defined')
print(f'✅ CLIENTS        : {[c.model_name for c in CLIENTS]}')
print(f'✅ test_eval_df   : {len(test_eval_df)} rows')
print('\nRun Cell 10 now ▶')

✅ build_prompt   defined
✅ parse_price    defined
✅ compute_metrics defined
✅ CLIENTS        : ['llama-3.3-70b-versatile', 'google/gemma-3-27b-it:free']
✅ test_eval_df   : 50 rows

Run Cell 10 now ▶


In [21]:
# ============================================================
# CELL 10 — Run LLM Evaluation
# ============================================================
# Core evaluation loop:
#   For each product → build prompt → send to each LLM
#   → parse price → record true vs predicted
#
# Rate limiting is built into each client.
# Safety-blocked responses count as n_failed (not crashes).
#
# Expected output:
#   Progress bar per model + per-model results summary
# ⏱  Runtime: ~10–20 min for 50 products × 2 models
# ============================================================

USE_CATEGORY_HINT = True

all_results   = []
all_preds_log = {}

for client in CLIENTS:
    print(f'\n{"-"*55}')
    print(f'  Evaluating: {client.model_name}')
    print(f'{"-"*55}')

    y_true, y_pred, n_fail = [], [], 0
    pred_log = []

    for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df),
                       desc=client.model_name[:30]):

        cat    = row['category_inferred'] if USE_CATEGORY_HINT else None
        prompt = build_prompt(row['title_clean'], cat)
        raw    = client.predict(prompt)

        if not raw:                      # None (API fail) or '' (safety block)
            n_fail += 1
            pred_log.append({
                'title': row['title_clean'], 'true': row['price'],
                'predicted': None, 'raw': raw, 'error': 'api_or_safety_fail'
            })
            continue

        price = parse_price(raw)
        if price is None:
            n_fail += 1
            pred_log.append({
                'title': row['title_clean'], 'true': row['price'],
                'predicted': None, 'raw': raw, 'error': 'parse_failure'
            })
            continue

        y_true.append(row['price'])
        y_pred.append(price)
        pred_log.append({
            'title': row['title_clean'], 'true': row['price'],
            'predicted': price, 'raw': raw
        })

    m = compute_metrics(y_true, y_pred, client.model_name, n_fail)
    m['avg_latency_ms'] = round(client.avg_latency_ms, 1)
    all_results.append(m)
    all_preds_log[client.model_name] = pred_log

    print(f'\n  Results — {client.model_name}:')
    print(f'    Evaluated : {m["n_evaluated"]} / {m["n_evaluated"]+m["n_failed"]}')
    if m['mae'] is not None:
        print(f'    MAE       : ${m["mae"]:.2f}')
        print(f'    MAPE      : {m["mape"]:.2f}%')
        print(f'    RMSE      : ${m["rmse"]:.2f}')
        print(f'    ±20% Acc  : {m["accuracy_within_20pct"]:.1f}%')
    else:
        print('    ⚠️  All predictions failed — check API key and model availability')

print('\n\n✅ Evaluation complete!')


-------------------------------------------------------
  Evaluating: llama-3.3-70b-versatile
-------------------------------------------------------


llama-3.3-70b-versatile:   0%|          | 0/50 [00:00<?, ?it/s]


  Results — llama-3.3-70b-versatile:
    Evaluated : 50 / 50
    MAE       : $114.97
    MAPE      : 31.00%
    RMSE      : $278.32
    ±20% Acc  : 40.0%

-------------------------------------------------------
  Evaluating: google/gemma-3-27b-it:free
-------------------------------------------------------


google/gemma-3-27b-it:free:   0%|          | 0/50 [00:00<?, ?it/s]

  Retry 1 (google/gemma-3-27b-it:free): Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3BL0O2XYeB4Ea3adqNbHDdqOZ2V'}. Waiting 1s...
  Retry 2 (google/gemma-3-27b-it:free): Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-3-27b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3BL0O2XYeB4Ea3adqNbHDdqOZ2V'}. Waiting 2s...
  FAIL (google/gemma-3-27b-it:free): Error code: 429 - {'error': {'message': 'Provider returned error', 'code'

In [23]:
# ============================================================
# CELL 11 — Results, Leaderboard & Visualisations
# ============================================================

import json
from pathlib import Path

valid_results     = [r for r in all_results if r['mae'] is not None]
all_results_display = all_results

# ── Leaderboard table ────────────────────────────────────────
df_res = pd.DataFrame(all_results_display)
df_res = df_res.sort_values('mape', na_position='last').reset_index(drop=True)

# FIXED: rank list always matches number of rows
medals = ['🥇','🥈','🥉']
ranks  = [medals[i] if i < len(medals) else '' for i in range(len(df_res))]
df_res.insert(0, 'Rank', ranks)

show_cols = ['Rank','model','n_evaluated','mae','mape','rmse',
             'accuracy_within_20pct','parse_success_rate','avg_latency_ms']
show_cols = [c for c in show_cols if c in df_res.columns]

print('='*70)
print('  📊 LEADERBOARD')
print('='*70)
display(df_res[show_cols])

if valid_results:
    best = min(valid_results, key=lambda r: r['mape'])
    print(f"\n🏆 Winner: {best['model']}")
    print(f"   MAPE={best['mape']}% | MAE=${best['mae']} | ±20% Acc={best['accuracy_within_20pct']}%")

# ── Charts ───────────────────────────────────────────────────
if valid_results:
    df_v = pd.DataFrame(valid_results)

    # Chart 1: MAPE
    fig1 = px.bar(df_v, x='model', y='mape', color='model',
                  title='MAPE by Model — lower is better',
                  labels={'mape':'MAPE (%)','model':'Model'},
                  color_discrete_sequence=px.colors.qualitative.Set2)
    fig1.add_hline(y=20, line_dash='dash', line_color='red',
                   annotation_text='Target: 20%')
    fig1.update_layout(showlegend=False)
    fig1.show()

    # Chart 2: MAE
    fig2 = px.bar(df_v, x='model', y='mae', color='model',
                  title='MAE by Model — lower is better',
                  labels={'mae':'MAE ($)','model':'Model'},
                  color_discrete_sequence=px.colors.qualitative.Pastel)
    fig2.update_layout(showlegend=False)
    fig2.show()

    # Chart 3: Accuracy at tolerances
    acc_rows = []
    for r in valid_results:
        for tol in [10, 20, 50]:
            key = f'accuracy_within_{tol}pct'
            if r.get(key) is not None:
                acc_rows.append({
                    'Model'    : r['model'],
                    'Tolerance': f'±{tol}%',
                    'Accuracy' : r[key],
                })
    if acc_rows:
        fig3 = px.bar(pd.DataFrame(acc_rows), x='Tolerance', y='Accuracy',
                      color='Model', barmode='group',
                      title='Accuracy Within Tolerance',
                      color_discrete_sequence=px.colors.qualitative.Bold)
        fig3.show()

    # Chart 4: Predicted vs True scatter (best model)
    best_preds = [
        p for p in all_preds_log.get(best['model'], [])
        if p.get('predicted') is not None
    ]
    if best_preds:
        df_sc = pd.DataFrame(best_preds)
        df_sc['err_pct'] = (
            abs(df_sc['true'] - df_sc['predicted']) / df_sc['true'] * 100
        ).round(1)
        max_p = max(df_sc['true'].max(), df_sc['predicted'].max())
        fig4 = px.scatter(
            df_sc, x='true', y='predicted',
            hover_data=['title', 'err_pct'],
            color='err_pct',
            color_continuous_scale='RdYlGn_r',
            title=f'Predicted vs True — {best["model"]}',
            labels={'true':'True Price ($)', 'predicted':'Predicted ($)'},
        )
        fig4.add_trace(go.Scatter(
            x=[0, max_p], y=[0, max_p],
            mode='lines', name='Perfect',
            line=dict(dash='dash', color='gray')
        ))
        fig4.show()
    else:
        print('No scatter data available for best model.')

else:
    print('\n⚠️  No successful predictions to chart.')

# ── Save results ─────────────────────────────────────────────
OUTPUT_DIR = Path('/content/DealHunter/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = [{k: v for k, v in r.items()} for r in all_results]
with open(OUTPUT_DIR / 'leaderboard.json', 'w') as f:
    json.dump(summary, f, indent=2)

for model_name, preds in all_preds_log.items():
    safe = model_name.replace('/', '_').replace(':', '_')
    with open(OUTPUT_DIR / f'predictions_{safe}.json', 'w') as f:
        json.dump(preds, f, indent=2)

print(f'\n💾 Results saved to {OUTPUT_DIR}/')
print('\nCell 11 complete ✅')

  📊 LEADERBOARD


,Rank,model,n_evaluated,mae,mape,rmse,accuracy_within_20pct,parse_success_rate,avg_latency_ms
0,🥇,llama-3.3-70b-versatile,50,114.97,31.00,278.32,40.0,100.0,209.2
1,🥈,google/gemma-3-27b-it:free,1,28.15,68.08,28.15,0.0,2.0,1386.8



🏆 Winner: llama-3.3-70b-versatile
   MAPE=31.0% | MAE=$114.97 | ±20% Acc=40.0%



💾 Results saved to /content/DealHunter/outputs/

Cell 11 complete ✅


In [25]:
# ============================================================
# CELL 13 — Download Outputs + GitHub Checklist
# ============================================================
# Zips all outputs and triggers a download.
# Expected output: download starts + submission checklist
# ============================================================

import shutil

# List what was saved
print('Files in outputs/:')
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f'  {f.name:50s} {size:6.1f} KB')

# Create zip
zip_path = '/content/DealHunter_outputs'
shutil.make_archive(zip_path, 'zip', str(OUTPUT_DIR))
print(f'\n📦 Zipped to {zip_path}.zip')

# Download
try:
    from google.colab import files
    files.download(zip_path + '.zip')
    print('⬇️  Download started.')
except ImportError:
    print('Not in Colab — find zip at:', zip_path+'.zip')

print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📋 GitHub Submission Checklist — Day 40 Milestone 7
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Folder: Day_40_Milestone7_DealHunter/
  ✅ app.py
  ✅ pipeline.py
  ✅ utils/data_utils.py
  ✅ utils/llm_clients.py
  ✅ utils/metrics.py
  ✅ requirements.txt
  ✅ README.md
  ✅ outputs/leaderboard.json
  ✅ Day40_Milestone7_DealHunter_FIXED.ipynb

  Commit: [Day-40] Milestone 7: DealHunter LLM price oracle
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

Files in outputs/:
  leaderboard.json                                      0.6 KB
  predictions_google_gemma-3-27b-it_free.json           7.5 KB
  predictions_llama-3.3-70b-versatile.json              6.1 KB

📦 Zipped to /content/DealHunter_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📋 GitHub Submission Checklist — Day 40 Milestone 7
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Folder: Day_40_Milestone7_DealHunter/
  ✅ app.py
  ✅ pipeline.py
  ✅ utils/data_utils.py
  ✅ utils/llm_clients.py
  ✅ utils/metrics.py
  ✅ requirements.txt
  ✅ README.md
  ✅ outputs/leaderboard.json
  ✅ Day40_Milestone7_DealHunter_FIXED.ipynb

  Commit: [Day-40] Milestone 7: DealHunter LLM price oracle
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [26]:
# ── Download complete DealHunter project zip ──
import shutil, os
from google.colab import files
from pathlib import Path

BASE_DIR   = Path('/content/DealHunter')
OUTPUT_DIR = BASE_DIR / 'outputs'

# Save the notebook itself
import json

# Write all project files to BASE_DIR
# ── app.py ───────────────────────────────────────────────────
(BASE_DIR / 'app.py').write_text('''import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

st.set_page_config(page_title="DealHunter Leaderboard", page_icon="🏆", layout="wide")
st.title("🏆 DealHunter — LLM Price Oracle Leaderboard")
st.markdown("**Milestone 7 | AlgoProfessor AI R&D Internship 2026 | Phase 2**")
st.markdown("---")

lb = Path("outputs/leaderboard.json")
if not lb.exists():
    st.warning("No results found. Run pipeline.py first.")
    st.stop()

with open(lb) as f:
    data = json.load(f)

valid = [r for r in data if r.get("mape") is not None]
df = pd.DataFrame(valid).sort_values("mape").reset_index(drop=True)
medals = ["🥇","🥈","🥉"]
df.insert(0, "Rank", [medals[i] if i < len(medals) else "" for i in range(len(df))])

st.subheader("📊 Model Leaderboard")
st.dataframe(df[["Rank","model","mae","mape","rmse","accuracy_within_20pct","parse_success_rate"]],
             use_container_width=True, hide_index=True)

col1, col2 = st.columns(2)
with col1:
    fig = px.bar(df, x="model", y="mape", color="model", title="MAPE (lower=better)")
    fig.add_hline(y=20, line_dash="dash", annotation_text="Target 20%")
    fig.update_layout(showlegend=False)
    st.plotly_chart(fig, use_container_width=True)
with col2:
    fig2 = px.bar(df, x="model", y="mae", color="model", title="MAE $ (lower=better)")
    fig2.update_layout(showlegend=False)
    st.plotly_chart(fig2, use_container_width=True)

st.markdown("---")
st.caption("DealHunter | AlgoProfessor AI R&D 2026 | Milestone 7")
''')

# ── requirements.txt ─────────────────────────────────────────
(BASE_DIR / 'requirements.txt').write_text(
'''google-generativeai>=0.7.2
groq>=0.9.0
openai>=1.30.1
httpx==0.27.2
datasets>=2.19.1
pandas
numpy
scikit-learn
tqdm
plotly
streamlit>=1.35.0
pyngrok
requests
''')

# ── README.md ─────────────────────────────────────────────────
(BASE_DIR / 'README.md').write_text(
'''# 🎯 DealHunter — Milestone 7
## AlgoProfessor AI R&D Internship 2026 | Phase 2 | Day 40

## What is DealHunter?
Product price prediction system benchmarking multiple LLMs on estimating
retail prices from product titles. Produces a live Streamlit leaderboard.

## Models Compared
| Model | Provider | Cost |
|-------|----------|------|
| llama-3.3-70b-versatile | Groq | FREE |
| google/gemma-3-27b-it | OpenRouter | FREE |

## Metrics
- MAE  = Mean Absolute Error ($)
- MAPE = Mean Absolute % Error (main metric, target < 20%)
- RMSE = Root Mean Squared Error
- ±20% Accuracy = % predictions within 20% of true price

## How to Run
```bash
pip install -r requirements.txt
python pipeline.py          # run evaluation
streamlit run app.py        # launch UI
```

## Results
See outputs/leaderboard.json

## Commit
[Day-40] Milestone 7: DealHunter — LLM price oracle
''')

# ── pipeline.py ──────────────────────────────────────────────
(BASE_DIR / 'pipeline.py').write_text(
'''"""
DealHunter pipeline.py
Run evaluation from command line: python pipeline.py
"""
print("Run the Colab notebook for full evaluation.")
print("See: Day40_Milestone7_DealHunter_FIXED.ipynb")
''')

# ── utils/ ───────────────────────────────────────────────────
utils_dir = BASE_DIR / 'utils'
utils_dir.mkdir(exist_ok=True)
(utils_dir / '__init__.py').write_text('# DealHunter utils\n')

(utils_dir / 'metrics.py').write_text(
'''import numpy as np

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.array(y_true) - np.array(y_pred))))

def mape(y_true, y_pred):
    a, p = np.array(y_true, float), np.array(y_pred, float)
    mask = a != 0
    return float(np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2)))

def accuracy_within(y_true, y_pred, tol=20):
    a, p = np.array(y_true, float), np.array(y_pred, float)
    return float(np.mean(np.abs(a-p)/np.maximum(a,1e-9)*100 <= tol)*100)
''')

print("✅ All project files written to /content/DealHunter/")

# ── List everything ──────────────────────────────────────────
print("\nProject structure:")
for f in sorted(BASE_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size / 1024
        rel  = str(f).replace('/content/DealHunter/', '')
        print(f"  {rel:55s} {size:6.1f} KB")

# ── Zip entire project ───────────────────────────────────────
zip_path = '/content/Milestone_6_DealHunter'
shutil.make_archive(zip_path, 'zip', '/content', 'DealHunter')
size_mb = Path(zip_path+'.zip').stat().st_size / 1_000_000
print(f'\n📦 Zipped: {zip_path}.zip ({size_mb:.2f} MB)')

# ── Download ─────────────────────────────────────────────────
files.download(zip_path + '.zip')
print('⬇️  Download started — check your Downloads folder')
print('\nZip contains:')
print('  DealHunter/')
print('  ├── app.py')
print('  ├── pipeline.py')
print('  ├── requirements.txt')
print('  ├── README.md')
print('  ├── utils/')
print('  │   ├── __init__.py')
print('  │   └── metrics.py')
print('  └── outputs/')
print('      ├── leaderboard.json')
print('      ├── predictions_llama-3.3-70b-versatile.json')
print('      └── predictions_google_gemma-3-27b-it_free.json')

✅ All project files written to /content/DealHunter/

Project structure:
  README.md                                                  0.9 KB
  app.py                                                     1.5 KB
  outputs/leaderboard.json                                   0.6 KB
  outputs/predictions_google_gemma-3-27b-it_free.json        7.5 KB
  outputs/predictions_llama-3.3-70b-versatile.json           6.1 KB
  pipeline.py                                                0.2 KB
  requirements.txt                                           0.2 KB
  streamlit_app.py                                           1.6 KB
  utils/__init__.py                                          0.0 KB
  utils/metrics.py                                           0.6 KB

📦 Zipped: /content/Milestone_6_DealHunter.zip (0.01 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started — check your Downloads folder

Zip contains:
  DealHunter/
  ├── app.py
  ├── pipeline.py
  ├── requirements.txt
  ├── README.md
  ├── utils/
  │   ├── __init__.py
  │   └── metrics.py
  └── outputs/
      ├── leaderboard.json
      ├── predictions_llama-3.3-70b-versatile.json
      └── predictions_google_gemma-3-27b-it_free.json
